# Smart Traffic Light: Vehicle Detection, Queue Estimation and Adaptive Signal Control

**Capstone Project - AI/ML Fundamentals - Individual Project Track**

**Student:** Ruxsora Ahmedova

## 1. Problem statement

Traffic lights at most intersections run on a fixed timer. The green light stays on for a preset number of seconds no matter how many vehicles are actually waiting. This causes long queues during peak hours and wasted green time when the road is nearly empty. When congestion becomes severe, traffic police are often called to direct vehicles by hand.

**Stakeholder:** municipal traffic management departments and drivers waiting at signalised intersections.

**ML task:** object detection (locate and classify vehicles in a video frame) combined with multi-object tracking (follow each vehicle across frames to measure how long it has been stationary).

**Input:** a single RGB video frame from a traffic camera.

**Output:** bounding boxes with vehicle class and confidence, plus derived values - number of vehicles waiting, estimated waiting time, and a recommended green-light duration.

**Success criteria:** the fine-tuned detector outperforms a non-ML baseline on the detection task, and the full pipeline produces a signal decision on a video the model has never seen.

**Scope:** this is a prototype running on recorded video. It is not connected to a real traffic signal controller.

## 2. Pipeline

1. Baseline: vehicle detection without machine learning (background subtraction)
2. Main model: YOLOv8 fine-tuned on a traffic intersection dataset
3. Comparison of approaches on held-out data
4. Tracking: ByteTrack assigns a persistent ID to each vehicle across frames
5. Queue analysis: count stationary vehicles and measure waiting time
6. Decision logic: threshold rule that extends or shortens the green light
7. Evaluation on unseen data and error analysis
8. Responsible AI considerations and limitations

## How to run this notebook

Run the cells **in order, one at a time**. Do not use "Run all" - if something fails you want to know exactly which step it was.

Before starting: `Runtime -> Change runtime type -> T4 GPU -> Save`.


## Step 0. Install dependencies

Takes 1-2 minutes. A lot of installation text will scroll past - that is normal. Wait for the confirmation line at the end.


In [ ]:
!pip install -q ultralytics yt-dlp opencv-python-headless roboflow pandas matplotlib pyyaml

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os, glob, time, json, hashlib, yaml, shutil
from collections import Counter

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Training will be very slow.")
    print("Fix: Runtime -> Change runtime type -> T4 GPU -> Save, then re-run this cell.")

print("Setup complete")


## Step 1. Load the dataset

**Dataset:** Traffic Intersection Vehicle Detection (author: VAI, Roboflow Universe)
Annotated images of vehicles at road intersections, licence CC BY 4.0.
https://universe.roboflow.com/vai/traffic-intersection-vehicle-detection

**Why fine-tuning instead of training from scratch:** YOLOv8 is already pretrained on COCO, a large general-purpose dataset that includes vehicles. Fine-tuning adapts those existing weights to this specific camera angle and image style, which needs far less data and training time than starting from random weights. This is called transfer learning.

To run this cell you need a free Roboflow API key: roboflow.com -> Settings -> API Key.


In [ ]:
from roboflow import Roboflow

API_KEY = "PASTE_YOUR_API_KEY_HERE"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("vai").project("traffic-intersection-vehicle-detection")

# Try the newest available version first and fall back if it does not exist.
dataset = None
for v in [4, 3, 2, 1]:
    try:
        dataset = project.version(v).download("yolov8")
        print("Downloaded dataset version", v)
        break
    except Exception as e:
        print("Version", v, "unavailable:", str(e)[:100])

if dataset is None:
    raise RuntimeError("Could not download any dataset version. Check your API key and internet connection.")

print("Dataset location:", dataset.location)


In [ ]:
# Roboflow sometimes writes relative paths into data.yaml that break inside Colab.
# Rewrite them as absolute paths so training cannot silently read the wrong folder.
yaml_path = os.path.join(dataset.location, 'data.yaml')

with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)

for split in ['train', 'val', 'test']:
    split_dir = os.path.join(dataset.location, split if split != 'val' else 'valid', 'images')
    if os.path.exists(split_dir):
        data_cfg[split] = split_dir
    elif split in data_cfg:
        del data_cfg[split]

with open(yaml_path, 'w') as f:
    yaml.safe_dump(data_cfg, f)

HAS_TEST_SPLIT = 'test' in data_cfg
EVAL_SPLIT = 'test' if HAS_TEST_SPLIT else 'val'

print("data.yaml paths fixed")
print("Classes:", data_cfg['names'])
print("Evaluation split to be used for final metrics:", EVAL_SPLIT)
if not HAS_TEST_SPLIT:
    print("NOTE: this dataset version has no separate test split, so the validation split")
    print("is used for the final numbers. Mention this as a limitation in your README.")


## Step 2. Exploratory data analysis

Before training, check the size of each split and how balanced the classes are. Class imbalance matters here: if one class dominates, aggregate metrics can look good while the model performs poorly on the rarer classes.


In [ ]:
train_images = glob.glob(dataset.location + '/train/images/*')
val_images   = glob.glob(dataset.location + '/valid/images/*')
test_images  = glob.glob(dataset.location + '/test/images/*')

print("Train images:     ", len(train_images))
print("Validation images:", len(val_images))
print("Test images:      ", len(test_images))

class_counts = Counter()
for lf in glob.glob(dataset.location + '/train/labels/*.txt'):
    with open(lf) as f:
        for line in f:
            parts = line.split()
            if parts:
                class_counts[data_cfg['names'][int(parts[0])]] += 1

print("\nAnnotated objects per class (train split):")
for cls, cnt in class_counts.most_common():
    print("  {:<12} {}".format(cls, cnt))

if class_counts:
    biggest = class_counts.most_common()[0]
    smallest = class_counts.most_common()[-1]
    ratio = biggest[1] / max(smallest[1], 1)
    print("\nImbalance ratio (most common / least common): {:.1f}x".format(ratio))

plt.figure(figsize=(8, 4))
plt.bar(list(class_counts.keys()), list(class_counts.values()), color='steelblue')
plt.title('Class distribution in the training split')
plt.ylabel('Number of annotated objects')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('eda_class_balance.png', dpi=120)
plt.show()


**EDA observations:** _[write your own observations here - which classes dominate, how large each split is, and what that implies for the metrics]_

## Step 2b. Data audit: duplicate and leakage check

This dataset is built from frames extracted from traffic camera video. Consecutive frames of the same scene look almost identical, so if they end up in both the training and test splits the model can effectively memorise the scene and the test score will look better than the model really is. This is called data leakage.

Below we compute an MD5 hash of every image file and check whether any identical file appears in more than one split.


In [ ]:
def hash_file(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def collect_hashes(folder):
    return {hash_file(p): p for p in glob.glob(folder + '/images/*')}

train_hashes = collect_hashes(dataset.location + '/train')
valid_hashes = collect_hashes(dataset.location + '/valid')
test_hashes  = collect_hashes(dataset.location + '/test') if test_images else {}

overlap_report = pd.DataFrame([
    {'split_pair': 'train-valid', 'exact_duplicate_images': len(set(train_hashes) & set(valid_hashes))},
    {'split_pair': 'train-test',  'exact_duplicate_images': len(set(train_hashes) & set(test_hashes))},
    {'split_pair': 'valid-test',  'exact_duplicate_images': len(set(valid_hashes) & set(test_hashes))},
])

print(overlap_report.to_string(index=False))
overlap_report.to_csv('duplicate_and_group_check.csv', index=False)

if overlap_report['exact_duplicate_images'].sum() > 0:
    print("\nWARNING: exact duplicate images found across splits. This is data leakage.")
    print("Document it in the README and treat the reported metrics as optimistic.")
else:
    print("\nNo exact duplicates found across splits (by file hash).")
    print("Note: this does not rule out near-duplicate frames from the same video,")
    print("which are visually almost identical but differ by a few pixels.")


**Leakage check result:** _[write what the check returned and how it affects the interpretation of your final metrics]_

## Step 3. Baseline without machine learning

A baseline gives a reference point: it shows what is achievable without a neural network, so the value added by the model can be measured rather than assumed.

This baseline uses background subtraction (MOG2). The algorithm builds a model of the static background of the scene and treats anything that differs from it as a moving object. It is simple and interpretable, but it has a known weakness for this exact task: a vehicle that stops for a long time gradually becomes part of the background and disappears from the detections. Since the whole point of the project is measuring vehicles that are **standing still**, this weakness is the core argument for using a learned detector instead.


In [ ]:
def baseline_detect_video(video_path, max_frames=200, min_area=500):
    """Detect moving objects using background subtraction. No machine learning."""
    cap = cv2.VideoCapture(video_path)
    back_sub = cv2.createBackgroundSubtractorMOG2(history=200, varThreshold=40, detectShadows=True)

    counts_per_frame = []
    frame_idx = 0

    while cap.isOpened() and frame_idx < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        fg_mask = back_sub.apply(frame)
        _, fg_mask = cv2.threshold(fg_mask, 250, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        vehicle_like = [c for c in contours if cv2.contourArea(c) > min_area]
        counts_per_frame.append(len(vehicle_like))
        frame_idx += 1

    cap.release()
    return counts_per_frame

print("Baseline function ready. It is run on the demo video in Step 7b.")


## Step 4. Fine-tuning YOLOv8 and experiment tracking

Key terms:

- **Epoch** - one full pass of the model over the entire training set.
- **Loss** - a number measuring how wrong the model currently is. YOLO tracks several: box_loss (how accurate the boxes are), cls_loss (how accurate the class labels are), dfl_loss (how precise the box edges are). Lower is better.
- **Train / validation split** - the model learns from the training split; the validation split checks whether it generalises rather than memorises.
- **mAP@50** - mean Average Precision at IoU 0.5, the standard object detection metric. It combines how well the model finds objects and how accurately it places the boxes.
- **Transfer learning** - starting from weights pretrained on another dataset instead of from random values.

Three experiments are run below with different settings, so the final model is chosen from evidence rather than by default. Model selection uses **validation** metrics only; the test split stays untouched until Step 7.

**Time budget:** the settings below are deliberately modest so all three experiments finish in a reasonable time on a free Colab GPU. Raise `EPOCHS_MAIN` if you have time to spare - and if you do, say so in the README, because the epoch budget is a real constraint worth documenting.


In [ ]:
EPOCHS_SHORT = 5     # quick, low-resolution run
EPOCHS_MAIN  = 15    # main run - raise this if you have time

experiment_log = []

def run_experiment(name, epochs, imgsz, lr0=0.01, batch=16):
    print("\n=== Running experiment:", name, "===")
    model = YOLO('yolov8n.pt')
    start = time.time()

    model.train(
        data=yaml_path,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        lr0=lr0,
        patience=5,
        project='traffic_light_project',
        name=name,
        exist_ok=True,
        verbose=False
    )

    duration = time.time() - start
    metrics = model.val(split='val')

    experiment_log.append({
        'experiment': name,
        'epochs': epochs,
        'imgsz': imgsz,
        'lr0': lr0,
        'batch': batch,
        'val_mAP50': round(float(metrics.box.map50), 4),
        'val_mAP50_95': round(float(metrics.box.map), 4),
        'val_precision': round(float(metrics.box.mp), 4),
        'val_recall': round(float(metrics.box.mr), 4),
        'train_time_min': round(duration / 60, 1)
    })
    return model

run_experiment('exp1_short',     epochs=EPOCHS_SHORT, imgsz=416)
run_experiment('exp2_longer',    epochs=EPOCHS_MAIN,  imgsz=640)
run_experiment('exp3_lower_lr',  epochs=EPOCHS_MAIN,  imgsz=640, lr0=0.001)

exp_df = pd.DataFrame(experiment_log)
print("\n=== EXPERIMENT LOG ===")
print(exp_df.to_string(index=False))
exp_df.to_csv('experiment_log.csv', index=False)

best_row = exp_df.loc[exp_df['val_mAP50'].idxmax()]
print("\nHighest validation mAP@50:", best_row['experiment'], "-", best_row['val_mAP50'])


**Which experiment did you choose and why:** _[compare the val_mAP50 column against train_time_min, then explain your choice in your own words. The highest score is not automatically the right answer if it cost far more time for a tiny gain.]_


In [ ]:
BEST_EXPERIMENT = 'exp2_longer'   # change this to whichever experiment you chose

best_weights_path = 'traffic_light_project/' + BEST_EXPERIMENT + '/weights/best.pt'
if not os.path.exists(best_weights_path):
    raise FileNotFoundError("Weights not found at " + best_weights_path +
                            ". Check that BEST_EXPERIMENT matches a name in the experiment log.")

final_model = YOLO(best_weights_path)

os.makedirs('artifacts', exist_ok=True)
shutil.copy(best_weights_path, 'artifacts/best_model.pt')

print("Final model saved to artifacts/best_model.pt")
print("Model classes:", final_model.names)


## Step 5. Training curves

The loss curve shows the model actually learned rather than just producing output. If training loss keeps falling while validation loss rises, the model is overfitting - memorising the training images instead of generalising.


In [ ]:
results_csv = 'traffic_light_project/' + BEST_EXPERIMENT + '/results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

def pick_col(frame, must_contain, must_not_contain=()):
    """Find a column by keyword. Ultralytics renames columns between versions."""
    for c in frame.columns:
        lc = c.lower()
        if all(m in lc for m in must_contain) and not any(x in lc for x in must_not_contain):
            return c
    return None

col_train_loss = pick_col(df, ['train', 'box_loss'])
col_val_loss   = pick_col(df, ['val', 'box_loss'])
col_map50      = pick_col(df, ['map50'], ['95'])
col_epoch      = pick_col(df, ['epoch']) or df.columns[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if col_train_loss:
    axes[0].plot(df[col_epoch], df[col_train_loss], label='train box_loss')
if col_val_loss:
    axes[0].plot(df[col_epoch], df[col_val_loss], label='validation box_loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Box loss per epoch')
axes[0].legend()

if col_map50:
    axes[1].plot(df[col_epoch], df[col_map50], color='green', label='mAP@50')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('mAP@50')
axes[1].set_title('Validation mAP@50 per epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=120)
plt.show()

if col_map50:
    print("Final validation mAP@50:", round(float(df[col_map50].iloc[-1]), 4))


**What the curves show:** _[say whether the loss went down and stayed down, and whether validation loss started rising - that would indicate overfitting]_

## Step 6. Get a demo video

This video is a completely separate source. The model has never seen it during training or validation, so it is the strongest available evidence that the pipeline works on genuinely new data.

Two options below. Try Option A first. If YouTube blocks the download (this happens regularly in Colab), use Option B and upload a short video file from your computer.


In [ ]:
# OPTION A: download from YouTube
YOUTUBE_URL = "PASTE_YOUTUBE_URL_HERE"

VIDEO_PATH = "traffic_video.mp4"

!yt-dlp -f "best[height<=480]" -o {VIDEO_PATH} "{YOUTUBE_URL}"

if os.path.exists(VIDEO_PATH):
    print("Download succeeded:", VIDEO_PATH)
else:
    print("Download failed. Use Option B in the next cell instead.")


In [ ]:
# OPTION B: upload a video file manually (only run this if Option A failed)
# Uncomment the three lines below, run the cell, then choose a file from your computer.

# from google.colab import files
# uploaded = files.upload()
# VIDEO_PATH = list(uploaded.keys())[0]

print("Current video path:", VIDEO_PATH if 'VIDEO_PATH' in dir() else 'not set')


## Step 7. Evaluation on unseen data

The fine-tuned model is evaluated on the held-out split, which was not used for training or for choosing between experiments.

**One important caveat about comparing against the pretrained model.** The pretrained YOLOv8 was trained on COCO, where the class numbering is completely different from this dataset (in COCO a car is class 2; here it may be class 0). When the pretrained model is scored against these labels, the class IDs do not line up, so its mAP will look close to zero. That number is therefore **not** a fair measure of how well the pretrained model sees vehicles - it mostly measures the label mismatch. It is reported below for transparency, but the honest comparison for this project is against the background-subtraction baseline in Step 7b.


In [ ]:
test_metrics_finetuned = final_model.val(data=yaml_path, split=EVAL_SPLIT)

pretrained_model = YOLO('yolov8n.pt')
test_metrics_pretrained = pretrained_model.val(data=yaml_path, split=EVAL_SPLIT)

comparison = pd.DataFrame([
    {
        'model': 'YOLOv8 pretrained (COCO classes, not aligned)',
        'mAP50': round(float(test_metrics_pretrained.box.map50), 4),
        'precision': round(float(test_metrics_pretrained.box.mp), 4),
        'recall': round(float(test_metrics_pretrained.box.mr), 4),
    },
    {
        'model': 'YOLOv8 fine-tuned (final model)',
        'mAP50': round(float(test_metrics_finetuned.box.map50), 4),
        'precision': round(float(test_metrics_finetuned.box.mp), 4),
        'recall': round(float(test_metrics_finetuned.box.mr), 4),
    },
])

print("Evaluated on split:", EVAL_SPLIT)
print(comparison.to_string(index=False))
comparison.to_csv('model_comparison.csv', index=False)


## Step 7b. Baseline comparison on video

The background-subtraction baseline works on video rather than on annotated still images, so it is compared here on the demo clip.


In [ ]:
baseline_counts = baseline_detect_video(VIDEO_PATH, max_frames=200)

print("Baseline (background subtraction) on the demo video:")
print("  Average moving regions per frame:", round(float(np.mean(baseline_counts)), 2))
print("  Maximum in a single frame:       ", max(baseline_counts))
print()
print("What the baseline cannot do:")
print("  - it detects motion, not vehicles, so shadows and pedestrians also count")
print("  - it cannot classify what it found")
print("  - it cannot follow an individual vehicle between frames")
print("  - vehicles that stop for long absorb into the background and vanish")
print()
print("The last point matters most here: the project needs to count vehicles that are")
print("standing still, which is exactly the case the baseline handles worst.")


**Comparison in your own words:** _[copy the numbers into the README table, then explain what the fine-tuned model achieved and why the baseline is not sufficient for measuring a queue]_

## Step 8. Error analysis

Metrics alone do not show *how* a model fails. Below the final model is run on several held-out images so specific failure cases can be identified.


In [ ]:
sample_source = test_images if test_images else val_images
sample_imgs = sample_source[:8]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, img_path in zip(axes.flatten(), sample_imgs):
    result = final_model.predict(img_path, conf=0.25, verbose=False)[0]
    ax.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(os.path.basename(img_path)[:24], fontsize=8)
    ax.axis('off')

for ax in axes.flatten()[len(sample_imgs):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig('error_analysis_samples.png', dpi=120)
plt.show()

print("Look for these failure types in the images above:")
print("  - missed vehicles (false negatives), often partly hidden behind other vehicles")
print("  - false detections (false positives), for example shadows, poles or signs")
print("  - class confusion, for example a bus labelled as a truck")
print("  - distant or very small vehicles being skipped entirely")


**Error analysis findings:** _[describe at least three concrete failure cases you can actually see in the images above, and say what could be done about each one]_

## Step 9. Detection and tracking on video

Detection works frame by frame and has no memory, so it cannot tell whether the car in this frame is the same car as in the previous one. ByteTrack solves this by assigning each vehicle a persistent ID. Once a vehicle has an ID, the change in its position between frames shows whether it is moving or waiting.

Note the class filtering below: the fine-tuned model uses **this dataset's** class numbering, not COCO's, so the vehicle class IDs are read from the model itself rather than hard-coded.


In [ ]:
VEHICLE_KEYWORDS = ['car', 'truck', 'bus', 'motorbike', 'motorcycle', 'van', 'vehicle', 'bike']

def vehicle_class_ids(model):
    """Read vehicle class IDs from the model itself so COCO and custom IDs never get mixed up."""
    ids = [i for i, name in model.names.items()
           if any(k in str(name).lower() for k in VEHICLE_KEYWORDS)]
    return ids if ids else None

VEHICLE_IDS = vehicle_class_ids(final_model)
print("Model classes:", final_model.names)
print("Treated as vehicles:", VEHICLE_IDS,
      [final_model.names[i] for i in VEHICLE_IDS] if VEHICLE_IDS else "(all classes)")


In [ ]:
MOVEMENT_THRESHOLD_PX = 5     # centre movement below this counts as stationary
WAITING_THRESHOLD_SEC = 2.0   # stationary longer than this counts as waiting
MAX_FRAMES = 600              # keeps runtime sane; raise for a longer demo


def process_traffic_video(video_path, model, output_path='output_annotated.mp4',
                          max_frames=MAX_FRAMES, save_preview=None):
    class_ids = vehicle_class_ids(model)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    stationary_frames = {}
    prev_positions = {}
    history = []
    preview_frames = []
    frame_idx = 0

    while cap.isOpened() and frame_idx < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        results = model.track(frame, persist=True, classes=class_ids, verbose=False)
        waiting_count = 0
        total_count = 0

        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xywh.cpu().numpy()
            track_ids = results[0].boxes.id.cpu().numpy().astype(int)
            total_count = len(track_ids)

            for box, tid in zip(boxes, track_ids):
                cx, cy = float(box[0]), float(box[1])

                if tid in prev_positions:
                    px, py = prev_positions[tid]
                    if np.hypot(cx - px, cy - py) < MOVEMENT_THRESHOLD_PX:
                        stationary_frames[tid] = stationary_frames.get(tid, 0) + 1
                    else:
                        stationary_frames[tid] = 0

                prev_positions[tid] = (cx, cy)

                if stationary_frames.get(tid, 0) / fps > WAITING_THRESHOLD_SEC:
                    waiting_count += 1

        annotated = results[0].plot()
        cv2.putText(annotated, 'Vehicles detected: ' + str(total_count), (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        cv2.putText(annotated, 'Waiting in queue: ' + str(waiting_count), (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)

        out.write(annotated)
        if save_preview and frame_idx % max(1, max_frames // 4) == 0 and len(preview_frames) < 4:
            preview_frames.append(annotated.copy())

        history.append({'frame': frame_idx, 'total': total_count, 'waiting': waiting_count})
        frame_idx += 1

    cap.release()
    out.release()

    if save_preview and preview_frames:
        fig, axes = plt.subplots(1, len(preview_frames), figsize=(5 * len(preview_frames), 4))
        if len(preview_frames) == 1:
            axes = [axes]
        for ax, fr in zip(axes, preview_frames):
            ax.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
            ax.axis('off')
        plt.tight_layout()
        plt.savefig(save_preview, dpi=120)
        plt.show()

    return pd.DataFrame(history), fps


history_df, fps = process_traffic_video(VIDEO_PATH, final_model,
                                        save_preview='demo_frames.png')
print("Processed", len(history_df), "frames. Video saved to output_annotated.mp4")


## Step 10. Signal control logic

This part is deliberately **not** a neural network. It is a threshold rule that takes the detector's output and decides how long the green light should stay on.

A rule was chosen over a learned policy for two reasons. First, no public dataset maps traffic conditions to correct signal timings, so there is nothing to train such a policy on without a traffic simulator. Second, a rule is interpretable and predictable, which matters for anything that could affect road safety. Learning the policy with reinforcement learning is listed as a possible extension, not part of this project.


In [ ]:
def decide_signal(waiting_count, base_green=30, extend_step=5, max_green=60, min_green=15):
    """Return the recommended green duration in seconds and the reason for it."""
    if waiting_count >= 8:
        return min(base_green + extend_step * 2, max_green), "Heavy queue - extend green"
    if waiting_count >= 4:
        return min(base_green + extend_step, max_green), "Moderate queue - extend green slightly"
    if waiting_count <= 1:
        return max(base_green - extend_step, min_green), "Almost empty - shorten green"
    return base_green, "Normal load - keep default timing"


# Sanity check the rule across the whole range before trusting it on real data
print("Rule behaviour check:")
for n in [0, 1, 2, 3, 4, 6, 8, 12]:
    g, reason = decide_signal(n)
    print("  {:>2} waiting -> {:>2}s   {}".format(n, g, reason))

window_frames = int(fps * 5)
recent_waiting = history_df['waiting'].tail(window_frames).mean()
green_time, decision = decide_signal(round(recent_waiting))

print("\n=== DECISION ON DEMO VIDEO ===")
print("Average vehicles waiting over the last 5 seconds:", round(recent_waiting, 1))
print("Decision:", decision)
print("Recommended green duration:", green_time, "seconds")

plt.figure(figsize=(12, 4))
plt.plot(history_df['frame'] / fps, history_df['total'], label='Vehicles detected')
plt.plot(history_df['frame'] / fps, history_df['waiting'], label='Vehicles waiting')
plt.xlabel('Time (seconds)')
plt.ylabel('Number of vehicles')
plt.title('Intersection load over time')
plt.legend()
plt.tight_layout()
plt.savefig('traffic_over_time.png', dpi=120)
plt.show()


## Step 11. Input validation

A usable pipeline should fail with a clear message rather than crash when given a missing or corrupted file.


In [ ]:
def validate_video(video_path):
    if not os.path.exists(video_path):
        raise FileNotFoundError("File not found: " + video_path)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        cap.release()
        raise ValueError("Could not open video (corrupted or unsupported format): " + video_path)

    ret, _ = cap.read()
    cap.release()
    if not ret:
        raise ValueError("Video contains no readable frames: " + video_path)

    return True


for bad_input in ['this_file_does_not_exist.mp4', 'README.md']:
    try:
        validate_video(bad_input)
        print(bad_input, "-> unexpectedly passed")
    except (FileNotFoundError, ValueError) as e:
        print("Handled correctly:", str(e)[:70])

validate_video(VIDEO_PATH)
print(VIDEO_PATH, "passed validation")


## Step 12. Reproducible inference demo

The model is loaded again **from the saved file**, not reused from the variable created during training. This shows that the saved artifact works on its own - which is what would happen if someone cloned the repository and ran the pipeline without training anything.


In [ ]:
inference_model = YOLO('artifacts/best_model.pt')

validate_video(VIDEO_PATH)
demo_history, demo_fps = process_traffic_video(VIDEO_PATH, inference_model,
                                               output_path='demo_output.mp4')

demo_waiting = demo_history['waiting'].tail(int(demo_fps * 5)).mean()
demo_green, demo_decision = decide_signal(round(demo_waiting))

print("=== FINAL DEMO (model loaded from file) ===")
print("Vehicles waiting:", round(demo_waiting))
print("Decision:", demo_decision)
print("Recommended green duration:", demo_green, "seconds")
print("Annotated video saved to demo_output.mp4")


## Step 13. Collect everything for the repository

This packages the model, tables and figures into one zip so they can be added to the GitHub repository in a single step.


In [ ]:
artifact_files = [
    'artifacts/best_model.pt',
    'experiment_log.csv',
    'model_comparison.csv',
    'duplicate_and_group_check.csv',
    'eda_class_balance.png',
    'training_curves.png',
    'error_analysis_samples.png',
    'traffic_over_time.png',
    'demo_frames.png',
    'demo_output.mp4',
]

present = [f for f in artifact_files if os.path.exists(f)]
missing = [f for f in artifact_files if not os.path.exists(f)]

print("Included:")
for f in present:
    print("  ", f, "({:.1f} MB)".format(os.path.getsize(f) / 1e6))
if missing:
    print("Missing (re-run the step that creates them):")
    for f in missing:
        print("  ", f)

!zip -q -r project_outputs.zip {" ".join(present)}
print("\nCreated project_outputs.zip")

from google.colab import files
files.download('project_outputs.zip')


## 14. Responsible AI

**Bias and representativeness**
The training images come from a limited set of intersections and conditions, so performance may drop at night, in rain or snow, or at camera angles unlike those in the dataset. The class distribution is uneven - cars greatly outnumber buses, trucks and motorbikes (see Step 2), so recall on the rarer classes is less reliable and aggregate mAP hides that.

**Privacy**
Traffic camera footage can contain licence plates and the faces of drivers and pedestrians. This prototype does not anonymise anything because it runs on publicly published data for a course project. A real deployment would need plates and faces blurred before any footage is stored or processed, along with a defined retention period.

**Safety**
The system produces a signal timing recommendation. A missed vehicle causes the queue to be underestimated and the green light to be cut short; a false detection extends it unnecessarily. Neither failure is acceptable in a live traffic controller without extensive testing and a fail-safe default timing. This prototype is not suitable for controlling real infrastructure.

**Appropriate use**
This project demonstrates an approach. It is not a validated product and should not be deployed on public roads without independent testing and approval from the relevant transport authority.

## 15. Conclusions

_[Write 3-5 sentences after everything has run: what was built, which model you selected and why, how it compared to the baseline, what the metrics were, and the single most useful thing you would improve with more time.]_
